In [19]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [20]:
file_path = "../data/Processed/furniture_returns_baseline.xlsx"

df = pd.read_excel(file_path)

print("Baseline dataset loaded successfully!")
print("Dataset shape:", df.shape)

Baseline dataset loaded successfully!
Dataset shape: (5057, 27)


In [21]:
feature_file_path = "../data/Processed/furniture_returns_featured.xlsx"

featured_df = pd.read_excel(feature_file_path)

print("Feature-engineered dataset loaded successfully!")
print("Dataset shape:", featured_df.shape)

Feature-engineered dataset loaded successfully!
Dataset shape: (5057, 23)


In [22]:
evaluation_file_path = (
    "../data/Processed/model_evaluation_results.xlsx"
)

metrics_df = pd.read_excel(
    evaluation_file_path,
    sheet_name="Classification Metrics",
    index_col=0
)

confusion_matrix_df = pd.read_excel(
    evaluation_file_path,
    sheet_name="Confusion Matrix",
    index_col=0
)

error_pairs = pd.read_excel(
    evaluation_file_path,
    sheet_name="Error Analysis"
)

feature_importance = pd.read_excel(
    evaluation_file_path,
    sheet_name="Feature Importance"
)

print("Model evaluation results loaded successfully!")

Model evaluation results loaded successfully!


In [23]:
print("=" * 60)
print("OVERALL PROJECT STATISTICS")
print("=" * 60)

print("\nTotal Returns:", len(df))

print("\nProducts:")
print(df["Product"].nunique())

print("\nReturn Reasons:")
print(df["Return Reason"].nunique())

print("\nInspection Outcomes:")
print(df["Inspection Outcome"].nunique())

print("\nBaseline Priority Distribution:")
print(df["Baseline Priority"].value_counts())

OVERALL PROJECT STATISTICS

Total Returns: 5057

Products:
8

Return Reasons:
6

Inspection Outcomes:
6

Baseline Priority Distribution:
Baseline Priority
Medium    2384
Low       2332
High       341
Name: count, dtype: int64


In [24]:
high_priority = df[
    df["Baseline Priority"] == "High"
]

print("=" * 60)
print("HIGH-PRIORITY RETURN PROFILE")
print("=" * 60)

print("\nHigh-priority returns:", len(high_priority))

print(
    "\nAverage product value: ₹",
    round(high_priority["Value ₹"].mean(), 2)
)

print(
    "\nAverage transit days:",
    round(high_priority["Transit Days"].mean(), 2)
)

print(
    "\nAverage SLA hours:",
    round(high_priority["SLA hrs"].mean(), 2)
)

HIGH-PRIORITY RETURN PROFILE

High-priority returns: 341

Average product value: ₹ 45190.06

Average transit days: 5.12

Average SLA hours: 40.96


In [25]:
high_priority_risk = (
    high_priority[
        [
            "Value Risk Score",
            "Severity Score",
            "Safety Score",
            "Condition Risk Score",
            "Transit Risk Score",
            "SLA Urgency Score"
        ]
    ]
    .mean()
    .round(2)
)

high_priority_risk

Value Risk Score        3.73
Severity Score          2.47
Safety Score            2.46
Condition Risk Score    2.26
Transit Risk Score      2.17
SLA Urgency Score       2.29
dtype: float64

In [26]:
product_analysis = (
    df.groupby("Product")
    .agg(
        Return_Count=("Return ID", "count"),
        Average_Value=("Value ₹", "mean"),
        Average_Transit_Days=("Transit Days", "mean"),
        High_Priority_Count=(
            "Baseline Priority",
            lambda x: (x == "High").sum()
        )
    )
    .sort_values(
        "Return_Count",
        ascending=False
    )
    .round(2)
)

product_analysis.head(20)

,Return_Count,Average_Value,Average_Transit_Days,High_Priority_Count
Product,,,,
chair,727,9051.92,7.63,0
bed,718,45414.63,6.14,138
bookshelf,715,12859.61,7.70,1
dining table,710,31743.53,9.22,49
sofa,691,45184.14,6.19,110
wardrobe,671,27048.89,6.34,32
coffee table,666,16049.55,3.32,3
unknown,159,23697.52,3.83,8


In [27]:
return_reason_analysis = (
    df.groupby("Return Reason")
    .agg(
        Return_Count=("Return ID", "count"),
        Average_Value=("Value ₹", "mean"),
        High_Priority_Count=(
            "Baseline Priority",
            lambda x: (x == "High").sum()
        )
    )
    .sort_values(
        "Return_Count",
        ascending=False
    )
    .round(2)
)

return_reason_analysis

,Return_Count,Average_Value,High_Priority_Count
Return Reason,,,
damaged,1030,27014.87,177
defective,994,25531.12,157
wrong product,988,26157.22,0
not as expected,946,26169.86,0
size issue,946,28398.94,0
unknown,153,26674.99,7


In [28]:
outcome_distribution = (
    df["Inspection Outcome"]
    .value_counts()
    .to_frame("Count")
)

outcome_distribution["Percentage"] = (
    outcome_distribution["Count"]
    / len(df) * 100
).round(2)

outcome_distribution

,Count,Percentage
Inspection Outcome,,
no defect,1955,38.66
minor damage,1847,36.52
repairable,674,13.33
major damage,244,4.82
scrap,189,3.74
unknown,148,2.93


In [29]:
top_features = feature_importance.head(10)

print("TOP 10 MODEL FEATURES")
print("=" * 60)

print(
    top_features.to_string(index=False)
)

TOP 10 MODEL FEATURES
                                      Feature  Importance
                    numerical__Severity Score    0.086534
                   numerical__Resale Before ₹    0.080193
categorical__Condition Hint_no visible damage    0.080131
                           numerical__Value ₹    0.078304
                   categorical__Severity_high    0.072033
                    categorical__Severity_low    0.066519
                categorical__Severity_unknown    0.060378
                 categorical__Severity_medium    0.048711
                      numerical__Transit Days    0.047753
              numerical__Condition Risk Score    0.038027


In [30]:
print("=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)

print(
    "Accuracy:",
    round(
        metrics_df.loc["accuracy", "precision"],
        3
    )
)

print(
    "Macro F1:",
    round(
        metrics_df.loc["macro avg", "f1-score"],
        3
    )
)

print(
    "Weighted F1:",
    round(
        metrics_df.loc["weighted avg", "f1-score"],
        3
    )
)

MODEL PERFORMANCE
Accuracy: 0.774
Macro F1: 0.494
Weighted F1: 0.785


In [31]:
major_damage_errors = (
    error_pairs[
        error_pairs["Actual Outcome"] == "major damage"
    ]["Count"]
    .sum()
)

scrap_errors = (
    error_pairs[
        error_pairs["Actual Outcome"] == "scrap"
    ]["Count"]
    .sum()
)

print("=" * 60)
print("CRITICAL MODEL ERRORS")
print("=" * 60)

print(
    "Major damage misclassifications:",
    major_damage_errors
)

print(
    "Scrap misclassifications:",
    scrap_errors
)

CRITICAL MODEL ERRORS
Major damage misclassifications: 26
Scrap misclassifications: 25


In [32]:
print("=" * 60)
print("KEY BUSINESS INSIGHTS")
print("=" * 60)

print(
    "\n1. The prioritization system ranks returns using "
    "multiple pre-inspection risk signals."
)

print(
    "\n2. Product value contributes to prioritization, "
    "but it is combined with severity, safety, condition, "
    "transit and SLA-related factors."
)

print(
    "\n3. High-priority returns represent cases requiring "
    "greater inspection attention."
)

print(
    "\n4. Model feature importance indicates that severity, "
    "condition information, product value and transit-related "
    "signals contribute substantially to prediction."
)

print(
    "\n5. Model errors remain for important inspection outcomes, "
    "so predictions should support rather than completely replace "
    "human inspection decisions."
)

print(
    "\n6. The system can help inspection teams organize the "
    "return queue and identify cases requiring earlier attention."
)

KEY BUSINESS INSIGHTS

1. The prioritization system ranks returns using multiple pre-inspection risk signals.

2. Product value contributes to prioritization, but it is combined with severity, safety, condition, transit and SLA-related factors.

3. High-priority returns represent cases requiring greater inspection attention.

4. Model feature importance indicates that severity, condition information, product value and transit-related signals contribute substantially to prediction.

5. Model errors remain for important inspection outcomes, so predictions should support rather than completely replace human inspection decisions.

6. The system can help inspection teams organize the return queue and identify cases requiring earlier attention.


In [33]:
recommendations = pd.DataFrame({
    "Area": [
        "Inspection Queue",
        "High-Risk Returns",
        "SLA Management",
        "Operational Planning",
        "Human Review",
        "Future Model Improvement"
    ],
    "Recommendation": [
        "Use the priority score to organize inspection order.",
        "Review high-priority returns earlier when operationally feasible.",
        "Monitor SLA urgency when allocating inspection capacity.",
        "Use transit and holding-cost information for operational planning.",
        "Use ML predictions as decision support with human review.",
        "Collect additional real operational outcomes and cost data for future model improvement."
    ]
})

recommendations

,Area,Recommendation
0,Inspection Queue,Use the priority score to organize inspection ...
1,High-Risk Returns,Review high-priority returns earlier when oper...
2,SLA Management,Monitor SLA urgency when allocating inspection...
3,Operational Planning,Use transit and holding-cost information for o...
4,Human Review,Use ML predictions as decision support with hu...
5,Future Model Improvement,Collect additional real operational outcomes a...


In [34]:
print("=" * 60)
print("PROJECT LIMITATIONS")
print("=" * 60)

print(
    "\n1. The dataset may not represent all real-world furniture returns."
)

print(
    "\n2. Holding-cost values used in trade-off analysis are illustrative assumptions."
)

print(
    "\n3. ML predictions are not a replacement for physical inspection."
)

print(
    "\n4. Some inspection outcome classes have limited representation, "
    "which affects model performance."
)

print(
    "\n5. Additional real operational data would improve future evaluation."
)

PROJECT LIMITATIONS

1. The dataset may not represent all real-world furniture returns.

2. Holding-cost values used in trade-off analysis are illustrative assumptions.

3. ML predictions are not a replacement for physical inspection.

4. Some inspection outcome classes have limited representation, which affects model performance.

5. Additional real operational data would improve future evaluation.


In [35]:
output_path = (
    "../data/Processed/"
    "business_insights_results.xlsx"
)

with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    product_analysis.to_excel(
        writer,
        sheet_name="Product Analysis"
    )

    return_reason_analysis.to_excel(
        writer,
        sheet_name="Return Reason Analysis"
    )

    outcome_distribution.to_excel(
        writer,
        sheet_name="Outcome Distribution"
    )

    top_features.to_excel(
        writer,
        sheet_name="Top Features",
        index=False
    )

    recommendations.to_excel(
        writer,
        sheet_name="Recommendations",
        index=False
    )

print("Business insights saved successfully!")
print(output_path)

Business insights saved successfully!
../data/Processed/business_insights_results.xlsx


In [36]:
print("=" * 60)
print("NOTEBOOK 08 COMPLETED")
print("=" * 60)

print("\nBusiness Analysis Completed:")
print("- Overall return statistics")
print("- High-priority return profile")
print("- Product-level analysis")
print("- Return-reason analysis")
print("- Inspection outcome analysis")
print("- Model feature importance")
print("- Model performance")
print("- Critical model errors")
print("- Business insights")
print("- Business recommendations")
print("- Project limitations")

print(
    "\nThe project has now completed the "
    "Notebook 01-08 analytical phase."
)

NOTEBOOK 08 COMPLETED

Business Analysis Completed:
- Overall return statistics
- High-priority return profile
- Product-level analysis
- Return-reason analysis
- Inspection outcome analysis
- Model feature importance
- Model performance
- Critical model errors
- Business insights
- Business recommendations
- Project limitations

The project has now completed the Notebook 01-08 analytical phase.
